# M23 lab solution: ReAct Agents & LangChain create_agent

**Brief.** ReAct research agent: give a `create_agent` agent your RAG retriever and a web-search tool, add a middleware that blocks off-topic questions, and compare its answers with your Part V system.

Run with the default environment: `uv run jupyter lab`. Needs `TAVILY_API_KEY` for the web-search tool.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
# The inline "Part V system", reused by the later labs: the handbook split as lab_kit.baseline splits it
# (recursive, 500 characters, 50 overlap), a FAISS vector retriever over those chunks, and an answer
# prompt that cites numbered passages and is allowed to say "I don't know".
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [
    Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
    for p in handbook
    for c in splitter.split_text(p["text"])
]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions from Larkspur Cloud customers using only the numbered handbook passages. "
               "Cite the passages you use as [1], [2]. If the passages do not contain the answer, "
               "reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata.get('source', '?')}) {d.page_content}" for i, d in enumerate(docs, 1))


def answer(question, docs, config=None):
    return (ANSWER_PROMPT | llm).invoke({"context": format_docs(docs), "question": question}, config=config).content


print(len(chunks), "chunks indexed")

In [ ]:
from pydantic import BaseModel, Field


class Grade(BaseModel):
    correct: bool = Field(description="True when the answer states every fact the reference answer gives for this question.")
    faithful: bool = Field(description="True when every claim in the answer is supported by the passages.")


JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You grade one answer from a customer-support RAG system. Judge it against the reference answer "
               "and the passages the system was given. Ignore citation markers such as [1]."),
    ("human", "Question: {question}\n\nReference answer: {reference}\n\nPassages:\n{context}\n\nAnswer to grade: {answer}"),
])
judge = JUDGE_PROMPT | llm.with_structured_output(Grade)


def grade(g, docs, reply):
    return judge.invoke({"question": g["question"], "reference": g["reference_answer"],
                         "context": format_docs(docs), "answer": reply})


def is_refusal(reply):
    return reply.strip().lower().startswith("i don't know")

In [ ]:
off_topic = load_jsonl(Path("../data/off_topic.jsonl"))
QUESTIONS = [by_id[q] for q in ["g01", "g02", "g03", "g08", "g17", "g19", "g22", "g24", "g26", "g28"]] + \
            [{**o, "type": "off_topic", "evidence_spans": [], "reference_answer": ""} for o in off_topic]
print(len(QUESTIONS), "questions:", [q["id"] for q in QUESTIONS])

## 1. The agent: retriever, web search and an off-topic guard

- `search_handbook` wraps the Part V retriever as a tool.
- `TavilySearch` searches the web, for questions the handbook does not cover.
- `off_topic_guard` is a `before_agent` middleware: one structured LLM call decides whether the question is about Larkspur Cloud or cloud hosting; if not, it writes a fixed reply and jumps to the end, so the agent never runs.
- `ModelCallLimitMiddleware(run_limit=6)` caps each run.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, before_agent
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field


@tool
def search_handbook(query: str) -> str:
    """Search the Larkspur Cloud customer handbook and return the most relevant passages with their source file."""
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in retriever.invoke(query))


class Topic(BaseModel):
    on_topic: bool = Field(description="True if the question is about Larkspur Cloud, its account, billing, servers or cloud hosting.")


topic_check = ChatPromptTemplate.from_messages([
    ("system", "Decide whether a customer question is in scope for Larkspur Cloud's support agent."),
    ("human", "{question}"),
]) | llm.with_structured_output(Topic)
REFUSAL = "I can only help with questions about Larkspur Cloud."


@before_agent(can_jump_to=["end"])
def off_topic_guard(state, runtime):
    question = state["messages"][-1].content
    if not topic_check.invoke({"question": question}).on_topic:
        return {"messages": [AIMessage(REFUSAL)], "jump_to": "end"}
    return None


agent = create_agent(
    model=llm,
    tools=[search_handbook, TavilySearch(max_results=3)],
    system_prompt=("You are Larkspur Cloud's support agent. Search the handbook first and answer from it, citing the source file. "
                   "Use web search only when the handbook has no answer, and say when an answer comes from the web."),
    middleware=[off_topic_guard, ModelCallLimitMiddleware(run_limit=6)],
)

## 2. Agent against the Part V system

Each question runs through the agent and through the inline Part V system (retrieve 4, answer with citations). In-corpus golden answers are graded by the judge for correctness; the out-of-corpus and off-topic rows record what each system did (refused, answered from the web, or answered anyway).

In [ ]:
rows = []
for q in QUESTIONS:
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q["question"]}]})
    agent_s = time.perf_counter() - t0
    msgs = result["messages"]
    calls = [c["name"] for m in msgs if isinstance(m, AIMessage) for c in (m.tool_calls or [])]
    agent_reply = msgs[-1].content if isinstance(msgs[-1].content, str) else str(msgs[-1].content)

    t0 = time.perf_counter()
    docs = retriever.invoke(q["question"])
    rag_reply = answer(q["question"], docs)
    rag_s = time.perf_counter() - t0

    row = {"id": q["id"], "type": q["type"], "blocked": agent_reply == REFUSAL, "tool_calls": ", ".join(calls) or "-",
           "agent_s": round(agent_s, 2), "rag_s": round(rag_s, 2), "rag_refused": is_refusal(rag_reply),
           "agent": agent_reply, "rag": rag_reply}
    if q["evidence_spans"]:
        row["agent_correct"] = grade(q, docs, agent_reply).correct
        row["rag_correct"] = grade(q, docs, rag_reply).correct
    rows.append(row)
    print(q["id"], "| blocked" if row["blocked"] else "", "| tools:", row["tool_calls"])
    print("   agent:", agent_reply[:140].replace("\n", " "))
    print("   rag:  ", rag_reply[:140].replace("\n", " "))

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
print("run:", time.strftime("%Y-%m-%d"), "| LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)
print(df[["id", "type", "blocked", "tool_calls", "agent_correct", "rag_correct", "rag_refused", "agent_s", "rag_s"]].to_string(index=False))
in_corpus = df.dropna(subset=["agent_correct"])
print("\ncorrect on", len(in_corpus), "in-corpus questions: agent", int(in_corpus["agent_correct"].sum()),
      "| Part V", int(in_corpus["rag_correct"].sum()))
print("off-topic blocked:", int(df[df["type"] == "off_topic"]["blocked"].sum()), "of", int((df["type"] == "off_topic").sum()))
print("golden questions blocked by mistake:", int(df[df["type"] != "off_topic"]["blocked"].sum()))

## What this shows

- The guard runs before the agent, so a blocked question costs one classification call and nothing else. Its two failure counts matter equally: off-topic questions it lets through, and customer questions it blocks.
- On the out-of-corpus golden questions the two systems behave differently by design: the Part V system says "I don't know", the agent may go to the web. Read those rows to decide which behaviour you want.
- The agent's latency and tool calls are the price of the loop; compare them with the single retrieval of the Part V system on the questions both answer correctly.